## Green's functions vs. finite elements in 2D

In 1D we built $u = G * f$ on the whole real line. Now we move to 2D, where the free-space Green's function is

$$
G(x) = \frac{1}{2\pi}\ln\|x\|, \qquad \Delta G = \delta .
$$

A finite element solver cannot work on all of $\mathbb{R}^2$: it needs a bounded domain and a boundary condition. Here we solve

$$
\Delta u = f \ \text{ in } [-L, L]^2, \qquad u = 0 \ \text{ on the boundary},
$$

We compare the two for two box sizes, then reuse the same solver to compute potential flows.

In [ ]:
import os
os.environ.setdefault("VISKEX_PYVISTA_BACKEND", "html")   # renders reliably inside VS Code notebooks

import numpy as np
import ufl
from mpi4py import MPI
from scipy.special import exp1
import dolfinx
from dolfinx.fem.petsc import LinearProblem
import viskex

from greens_fem_plots import eval_points, compare_along_x, plot_flow

### Exact solution

As in 1D we use a Gaussian forcing with total mass 1,

$$
f(x) = \frac{1}{2\pi\sigma^2} e^{-\|x\|^2 / 2\sigma^2}.
$$

After converting to polar, a solution to the convolution for this forcing can be found:

$$
u(r) = \frac{1}{4\pi}\left[\ln r^2 + E_1\!\left(\frac{r^2}{2\sigma^2}\right)\right].
$$
for $r != 0$ where $E_1$ is the exponential integral function, and $u(0) = \frac{1}{4\pi}\left(\ln 2\sigma^2 - \gamma\right)$, where $\gamma$ is the Euler–Mascheroni constant.

In [ ]:
sigma = 0.2

def u_exact(r):
    r2 = np.asarray(r, dtype=float)**2
    with np.errstate(divide='ignore', invalid='ignore'):
        val = (np.log(r2) + exp1(r2/(2*sigma**2)))/(4*np.pi)
    return np.where(r2 > 0, val, (np.log(2*sigma**2) - np.euler_gamma)/(4*np.pi))

def f_ufl(x):
    return ufl.exp(-ufl.dot(x, x)/(2*sigma**2))/(2*np.pi*sigma**2)

### The mesh

Finite elements chop the domain into small triangles. dolfinx can build a rectangle directly: give two opposite corners and the number of cells in each direction.

In [ ]:
domain = dolfinx.mesh.create_rectangle(MPI.COMM_WORLD, [[-1,-1],[1,1]],[8,8], dolfinx.mesh.CellType.triangle)
viskex.dolfinx.plot_mesh(domain)

### The finite element solve

On each triangle we approximate $u$ by a flat (linear) piece, so the approximate solution $u_h$ is a piecewise-linear "tent" surface: the 2D version of the hat functions from 1D. The unknowns are the values of $u_h$ at the mesh vertices.

To get one equation per unknown, multiply $\Delta u = f$ by a test function $v$ (also piecewise linear, zero on the boundary) and integrate by parts:

$$
\int_\Omega \nabla u \cdot \nabla v \, dx = -\int_\Omega f\, v\, dx \qquad \text{for every test function } v .
$$

FEniCSx turns this into a linear system $A\mathbf{u} = \mathbf{b}$ and solves it. The function below does all of this on the square $[-L, L]^2$. The forcing $f$ and the boundary values are inputs, so we can reuse it later.

In [ ]:
def solve_poisson(L, f, boundary_value, cells_per_unit=40):
    """Solve Δu = f on the square [-L, L]^2 with u = boundary_value on the edges."""
    # mesh: about 40 triangles per unit length in each direction
    n = int(2*L*cells_per_unit)
    domain = dolfinx.mesh.create_rectangle(MPI.COMM_WORLD, [[-L, -L], [L, L]], [n, n])
    V = dolfinx.fem.functionspace(domain, ("Lagrange", 1))      # piecewise-linear functions

    # boundary condition: u = boundary_value on every edge of the square
    u_bc = dolfinx.fem.Function(V)
    u_bc.interpolate(boundary_value)
    domain.topology.create_connectivity(1, 2)
    boundary_dofs = dolfinx.fem.locate_dofs_topological(V, 1, dolfinx.mesh.exterior_facet_indices(domain.topology))
    bc = dolfinx.fem.dirichletbc(u_bc, boundary_dofs)

    # weak form:  ∫ ∇u·∇v dx = -∫ f v dx
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(domain)
    a = ufl.inner(ufl.grad(u), ufl.grad(v))*ufl.dx
    rhs = -f(x)*v*ufl.dx

    problem = LinearProblem(a, rhs, bcs=[bc], petsc_options_prefix="poisson_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    return problem.solve()

## Example 1: the box $[-1, 1]^2$

In [ ]:
zero = lambda x: np.zeros_like(x[0])

u1 = solve_poisson(1.0, f_ufl, zero)
compare_along_x(u1, u_exact, "L = 1")

In [ ]:
viskex.dolfinx.plot_scalar_field(u1, "u", warp_factor=3.0, show_edges=False)

The finite element solution lands right on top of $G * f$. Is the boundary simply irrelevant?

No, this is a coincidence. Far from the source, $G * f \approx \frac{1}{2\pi}\ln r$, so on the boundary of the box $G * f$ is roughly $\frac{1}{2\pi}\ln L$. For $L = 1$ that is $\frac{1}{2\pi}\ln 1 = 0$: the boundary condition $u = 0$ happens to be (almost) exactly what $G * f$ would be there anyway.

## Example 2: the box $[-4, 4]^2$

In [ ]:
u4 = solve_poisson(4.0, f_ufl, zero)
compare_along_x(u4, u_exact, "L = 4")

offset = u_exact(0.0) - eval_points(u4, np.zeros((1, 2)))[0]
print(f"gap at the origin: {offset:.3f}     (1/2π) ln 4 = {np.log(4)/(2*np.pi):.3f}")

Now the two curves have the **same shape but are shifted** by about $\frac{1}{2\pi}\ln 4$.

The boundary condition $u = 0$ is wrong by about $\frac{1}{2\pi}\ln L$ all the way around the box, and that error spreads into the interior as a (nearly) constant shift. A bigger box makes the gap *bigger*, not smaller, because the 2D Green's function grows like $\ln r$ instead of decaying.

But a constant shift does not change $\nabla u$. If the quantity we care about is the gradient, the boundary matters much less. That is exactly the situation in potential flow.

## Potential flow

In an ideal fluid (no viscosity, no swirl) the velocity is the gradient of a potential, $\mathbf{v} = \nabla \phi$. Conservation of mass says

$$
\nabla \cdot \mathbf{v} = \Delta \phi = f ,
$$

where $f > 0$ is a **source** (fluid is injected) and $f < 0$ is a **sink** (fluid is removed). So the same Poisson solver computes flows.

Put a Gaussian source at $(-1, 0)$ and an equal sink at $(1, 0)$. The colour is the speed $|\mathbf{v}|$ and the white lines are streamlines, the paths fluid particles follow.

In [ ]:
d = 1.0
def source_sink(x):
    return f_ufl(x + ufl.as_vector([d, 0])) - f_ufl(x - ufl.as_vector([d, 0]))

phi = solve_poisson(3.0, source_sink, zero)
plot_flow(phi, sources=[(-d, 0)])

Fluid leaves the source and arcs over to the sink: the same picture as the field lines of a bar magnet. Because the total forcing is zero (as much fluid enters as leaves), this flow decays far away, so the box boundary disturbs it much less than it disturbed the single source in Example 2.

### Adding a uniform stream

Now set the boundary values to $\phi = Ux$ instead of $0$. On its own, $\phi = Ux$ is a uniform flow to the right with speed $U$.

In [ ]:
U = 0.2
phi = solve_poisson(3.0, source_sink, lambda x: U*x[0])
plot_flow(phi, sources=[(-d, 0)], inflow=True)

This is the **Rankine oval**. One closed streamline separates the fluid that travels from the source to the sink from the stream flowing past. The outside flow is exactly the flow around a solid oval body, so a source and a sink are enough to model flow past an obstacle. The dark spots at the front and back of the oval are **stagnation points**, where the stream and the source (or sink) cancel and the fluid is at rest.

**Questions to think about:**

1) What happens to the oval when $U$ increases? Try $U = 0.5$.
2) Remove the sink and keep the source and the stream. What shape do you get?
3) Example 1 matched because $\ln 1 = 0$. Would it still match if we measured lengths in centimetres instead of metres?
4) Why does the box boundary matter less for the source and sink pair than for the single source?